In [1]:
import os.path as op
import importlib

import numpy as np

from joblib import Parallel, delayed
from tqdm.notebook import tqdm
from time import perf_counter

import sys
sys.path.append("../..")

from bimodularity import dgsp
from bimodularity import data_load as dload
from bimodularity import bundle as b_utils

In [ ]:
path_to_brain_data = op.join("../../data", "brain")
path_to_derivatives = op.join(path_to_brain_data, "derivatives")

path_to_sc = op.join(path_to_derivatives, "structural_connectome")
path_to_ec = op.join(path_to_derivatives, "atlas_correspondence", "rDCM")
path_to_consensus = op.join(path_to_derivatives, "consensus_clustering")

# Consensus Clustering

In [ ]:
importlib.reload(b_utils)

all_scales = [2]

struct_type = ""
use_delay = False
k_threshold = 0
b_thresh = 0.5
slines_theshold = 5

n_trials = 50
n_init = 50
dthresh = 0
selected_sort = 0

undirected = False
nosquared = False

run_fix = False
if undirected:
    run_fix = True

all_n_kmeans = np.arange(10, 80, 1)
all_nvecs = [20]
all_gammas = [1, 2, 2.5, 3, 3.5, 4]
all_gammas = [1]

for gamma in all_gammas:
    for n_vec_max in all_nvecs:
        for scale in all_scales:
            k_matrix, labels, node_centers = dload.load_bundle_graph(path_to_data=path_to_sc,
                                                                    data_suffix="Laus2018_",
                                                                    scale=scale,
                                                                    b_prob_threshold=0,
                                                                    slines_theshold=slines_theshold,
                                                                    log_slines=False,
                                                                    normalize_slines=False,
                                                                    verbose=False)

            labels, k_matrix = b_utils.fix_thalamus(labels, matrix=k_matrix)
            k_matrix[k_matrix < b_thresh] = 0

            np.fill_diagonal(k_matrix, 0)

            if undirected:
                graph = ((k_matrix + k_matrix.T) / 2 > 0).astype(int)
            else:
                ec_mat = dload.get_ec_data(scale=scale, path_to_ec=path_to_ec, remove_neg=True, fix_thal=True, verbose=False)
                ec_mat /= ec_mat.max()
                ec_mat = ec_mat ** gamma

                dir_ratio = np.divide(ec_mat, (ec_mat + ec_mat.T),
                                    where=(ec_mat + ec_mat.T) > 1e-5,
                                    out=np.zeros_like(ec_mat))
                graph = (k_matrix > 0) * dir_ratio

            U, S, Vh = dgsp.sorted_SVD(dgsp.modularity_matrix(graph))
            V = Vh.T

            n_nodes = len(S)
            
            if nosquared:
                scale_factor = S[:n_vec_max]
            else:
                scale_factor = S[:n_vec_max] ** 2

            all_labels = np.zeros((len(all_n_kmeans), n_trials, np.sum(graph != 0)), dtype=np.int8)
            all_labels_fix = np.zeros((len(all_n_kmeans), n_trials, np.sum(graph != 0)), dtype=np.int8)

            cons_lab = np.zeros((len(all_n_kmeans), np.sum(graph != 0)), dtype=np.int8)
            cons_lab_fix = np.zeros((len(all_n_kmeans), np.sum(graph != 0)), dtype=np.int8)

            all_c_within = np.zeros(len(all_n_kmeans))
            all_c_between = np.zeros(len(all_n_kmeans))

            all_c_within_fix = np.zeros(len(all_n_kmeans))
            all_c_between_fix = np.zeros(len(all_n_kmeans))

            avg_cons = np.zeros((np.sum(graph != 0), np.sum(graph != 0)), dtype=int)
            avg_cons_fix = np.zeros((np.sum(graph != 0), np.sum(graph != 0)), dtype=int)
            reorder = False

            bm_fname = "_".join([
                f"brain_consensus-EC",
                f"scale{scale}",
                f"nvec{n_vec_max}" + "-NoSquared"*nosquared,
                f"trials{n_trials}",
                f"ninit{n_init}",
                f"kmeans{all_n_kmeans[0]}-{all_n_kmeans[-1]}",
                f"slines_thresh{slines_theshold}",
                f"ustruct{struct_type}"*(struct_type != ""),
                f"dthresh{dthresh}"*(dthresh != 0),
                f"gamma{gamma}"*(gamma != 1),
                f"selected{selected_sort}"*(selected_sort != 0)
                ])
            bm_fname += ".pkl"

            if undirected:
                bm_fname = "Undir_" + bm_fname

            if op.isfile(op.join(path_to_consensus, bm_fname)):
                data = dload.load(op.join(path_to_consensus, bm_fname))
                graph = data["graph"]
                all_n_kmeans = data["all_n_kmeans"]
                cons_lab = data["cons_lab"]
                cons_lab_fix = data["cons_lab_fix"]
                all_c_within = data["all_c_within"]
                all_c_between = data["all_c_between"]
                all_c_within_fix = data["all_c_within_fix"]
                all_c_between_fix = data["all_c_between_fix"]
                avg_cons = data["avg_cons"]
                avg_cons_fix = data["avg_cons_fix"]

                # # Override the anatomical bundle correspondence
                # edge_to_bundle_mat = b_utils.get_edge_to_bundle(graph, scale, labels, overlap_thresh=0.1)
                # dload.save(op.join(path_to_consensus, bm_fname),
                #             {"graph": graph,
                #             "all_n_kmeans": all_n_kmeans,
                #             "cons_lab": cons_lab,
                #             "cons_lab_fix": cons_lab_fix,
                #             "all_c_within": all_c_within,
                #             "all_c_between": all_c_between,
                #             "all_c_within_fix": all_c_within_fix,
                #             "all_c_between_fix": all_c_between_fix,
                #             "avg_cons": avg_cons,
                #             "avg_cons_fix": avg_cons_fix,
                #             "edge_to_bundle_mat": edge_to_bundle_mat})
            elif op.isfile(op.join(path_to_consensus, bm_fname.replace(".pkl", ".pkl.gz"))):
                print("Found gzipped version of the file! Skipping.")

                # os.system(f"gunzip -k {op.join(path_to_consensus, bm_fname.replace('.pkl', '.pkl.gz'))}")
                
                # data = dload.load(op.join(path_to_consensus, bm_fname))
                # graph = data["graph"]
                # all_n_kmeans = data["all_n_kmeans"]
                # cons_lab = data["cons_lab"]
                # cons_lab_fix = data["cons_lab_fix"]
                # all_c_within = data["all_c_within"]
                # all_c_between = data["all_c_between"]
                # all_c_within_fix = data["all_c_within_fix"]
                # all_c_between_fix = data["all_c_between_fix"]
                # avg_cons = data["avg_cons"]
                # avg_cons_fix = data["avg_cons_fix"]

                # os.system(f"rm {op.join(path_to_consensus, bm_fname.replace('.pkl', '.pkl.gz'))}")
            else:

                njobs = 1

                parallel = Parallel(n_jobs=np.min([14, n_trials]), return_as="generator_unordered")
                counter = tqdm(total=len(all_n_kmeans)*n_trials)

                t_cl = []
                t_co = []
                t_fast = []
                t_cs = []

                # for k_i, n_kmeans in tqdm(enumerate(all_n_kmeans), total=len(all_n_kmeans)):
                for k_i, n_kmeans in enumerate(all_n_kmeans):
                    t_1 = perf_counter()
                    gen = parallel(delayed(dgsp.edge_bicommunities)(
                        graph, U, V, 
                        n_vec_max,
                        method="kmeans",
                        n_kmeans=n_kmeans,
                        verbose=False,
                        max_k=50,
                        scale_S=scale_factor,
                        clust_only=True,
                        undirected=True,
                        kwargs_kmeans1={"n_init":n_init, "init":"random"},
                        kwargs_kmeans2={"n_init":1, "tol":1e-10},
                        dthresh=dthresh,
                        ) for _ in range(n_trials))
                    
                    for t, labs in enumerate(gen):
                        all_labels[k_i, t] = labs[0]
                        all_labels_fix[k_i, t] = labs[1]
                        counter.update(1)

                    t_2 = perf_counter()
                    t_cl.append(t_2 - t_1)
                    # print(f"Clustering took {t_cl[-1]:.2f} seconds (mean: {np.mean(t_cl):1.2f} s)")

                    cons, cons_lab[k_i] = dgsp.consensus_matrix(all_labels[k_i], reorder=reorder, consensus_labels=True, fast=True, njobs=njobs)
                    if run_fix:
                        cons_fix, cons_lab_fix[k_i] = dgsp.consensus_matrix(all_labels_fix[k_i], reorder=reorder, consensus_labels=True, fast=True, njobs=njobs)

                    t_3 = perf_counter()
                    t_co.append(t_3 - t_2)
                    # print(f"Consensus matrix computation took {t_co[-1]:.2f} seconds (mean: {np.mean(t_co):1.2f} s)")

                    avg_cons += cons
                    if run_fix:
                        avg_cons_fix += cons_fix

                    all_c_within[k_i], all_c_between[k_i] = dgsp.consensus_score(cons, cons_lab[k_i], is_sorted=not reorder)
                    if run_fix:
                        all_c_within_fix[k_i], all_c_between_fix[k_i] = dgsp.consensus_score(cons_fix, cons_lab_fix[k_i], is_sorted=not reorder)

                    t_4 = perf_counter()
                    t_cs.append(t_4 - t_3)
                    # print(f"Consensus score took {t_cs[-1]:.2f} seconds (mean: {np.mean(t_cs):1.2f} s)")

                avg_cons = avg_cons.astype(float) / (len(all_n_kmeans) * n_trials)
                if run_fix:
                    avg_cons_fix = avg_cons_fix.astype(float) / (len(all_n_kmeans) * n_trials)
                else:
                    avg_cons_fix = None

                print(f"Consensus Clustering Done")
                print(f"\t- Clustering took {np.mean(t_cl):1.2f}s on average")
                print(f"\t- Consensus Matrix took {np.mean(t_co):1.2f}s on average")
                print(f"\t- Consensus Score took {np.mean(t_cs):1.2f}s on average")

                counter.close()

                edge_to_bundle_mat = b_utils.get_edge_to_bundle(graph, scale, labels, overlap_thresh=0.1)

                dload.save(op.join(path_to_consensus, bm_fname),
                            {"graph": graph,
                            "all_n_kmeans": all_n_kmeans,
                            "cons_lab": cons_lab,
                            "cons_lab_fix": cons_lab_fix,
                            "all_c_within": all_c_within,
                            "all_c_between": all_c_between,
                            "all_c_within_fix": all_c_within_fix,
                            "all_c_between_fix": all_c_between_fix,
                            "avg_cons": avg_cons,
                            "avg_cons_fix": avg_cons_fix,
                            "edge_to_bundle_mat": edge_to_bundle_mat})